## Instalacion de dependencias

In [ ]:
%pip install pandas numpy scikit-learn lightgbm matplotlib

## Carga de librerias y archivos

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import lightgbm as lgb

# Cargar datos
os.chdir("C:\\Users\\Cristian David\\Desktop\\Todo\\U\\Modelos\\udea-ai-4-eng-20251-pruebas-saber-pro-colombia")
train = pd.read_csv("train.csv")

## Selecci?n de variables y divisi?n de datos

In [ ]:
categorical_features = [
    "ESTU_VALORMATRICULAUNIVERSIDAD",
    "ESTU_HORASSEMANATRABAJA",
    "FAMI_ESTRATOVIVIENDA",
    "FAMI_EDUCACIONPADRE",
    "FAMI_EDUCACIONMADRE",
    "ESTU_PRGM_DEPARTAMENTO",
    "FAMI_TIENEINTERNET",
    "ESTU_PAGOMATRICULAPROPIO",
    "ESTU_PRGM_ACADEMICO"
]

# Asegurar que las variables categ?ricas sean string y sin nulos
temp = train.copy()
for col in categorical_features:
    if col in temp.columns:
        temp[col] = temp[col].astype(str).fillna('missing')

y = temp["RENDIMIENTO_GLOBAL"]
X = temp.drop(columns=["RENDIMIENTO_GLOBAL", "ID", "PERIODO"])
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp)

## Procesamiento para LightGBM

In [ ]:
# Identificar correctamente las columnas num?ricas (excluyendo categ?ricas y asegurando que existan en X_train)
numeric_features = [col for col in X_train.select_dtypes(include=[np.number]).columns if col not in categorical_features]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
])

# Creaci?n del pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.1, random_state=42))
])
pipeline.fit(X_train, y_train)

# Evaluaci?n del modelo
y_val_pred = pipeline.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"Accuracy validaci?n: {val_accuracy:.4f}")
print(classification_report(y_val, y_val_pred))

y_test_pred = pipeline.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Accuracy test: {test_accuracy:.4f}")
print(classification_report(y_test, y_test_pred))


## Kaggle Submission

### Carga de datos

In [ ]:
test_data = pd.read_csv("test.csv")

In [ ]:
for col in categorical_features:
    if col in test_data.columns:
        test_data[col] = test_data[col].astype(str).fillna('missing')
X_test_kaggle = test_data.drop(columns=["ID", "PERIODO"], errors='ignore')
predictions = pipeline.predict(X_test_kaggle)
submission_df = test_data[["ID"]].copy()
submission_df["RENDIMIENTO_GLOBAL"] = predictions
submission_df.to_csv("submission_lightgbm.csv", index=False)
print("Archivo de submission generado: submission_lightgbm.csv")
print(submission_df.head())